In [ ]:
import os
from rosbag2_py import (
    SequentialReader,
    SequentialWriter,
    StorageOptions,
    ConverterOptions,
)
from rclpy.serialization import deserialize_message, serialize_message
from visualization_msgs.msg import Marker

In [ ]:
# Function to update IDs in the marker messages
def update_marker_id(marker_msg, id_offset):
    marker_msg.id += id_offset
    return marker_msg

In [ ]:
# Paths
input_folder = "<DATA_DIR>/sando/paper/global_planner_benchmarking/bags/baseline"
output_folder = os.path.join(input_folder, "processed")
os.makedirs(output_folder, exist_ok=True)

# Iterate through all folders in the input folder
folders = [
    f for f in os.listdir(input_folder) if os.path.isdir(os.path.join(input_folder, f))
]

id_offset = 0

for folder in folders:
    input_folder_path = os.path.join(input_folder, folder)
    db3_files = [f for f in os.listdir(input_folder_path) if f.endswith(".db3")]

    if not db3_files:
        print(f"No .db3 files found in folder: {folder}")
        continue

    db3_file = db3_files[0]  # Assuming one .db3 file per folder
    input_bag_path = os.path.join(input_folder_path, db3_file)
    output_bag_name = f"{folder}_shareable.db3"
    output_bag_path = os.path.join(output_folder, output_bag_name)

    # Set up reader and writer
    reader = SequentialReader()
    reader.open(
        StorageOptions(uri=input_bag_path, storage_id="sqlite3"),
        ConverterOptions(
            input_serialization_format="cdr", output_serialization_format="cdr"
        ),
    )

    writer = SequentialWriter()
    writer.open(
        StorageOptions(uri=output_bag_path, storage_id="sqlite3"),
        ConverterOptions(
            input_serialization_format="cdr", output_serialization_format="cdr"
        ),
    )

    # Register all topics for writing
    for topic_metadata in reader.get_all_topics_and_types():
        writer.create_topic(topic_metadata)

    while reader.has_next():
        (topic, data, timestamp) = reader.read_next()

        if topic == "/NX01/actual_traj":  # Replace with your actual Marker topic name
            marker_msg = deserialize_message(data, Marker)
            updated_marker_msg = update_marker_id(marker_msg, id_offset)
            serialized_message = serialize_message(updated_marker_msg)
            writer.write(topic, serialized_message, timestamp)
        # else:
        #     writer.write(topic, data, timestamp)

    # Get the last used id
    last_id = updated_marker_msg.id
    print(f"Last ID in {output_bag_name}: {last_id}")
    id_offset = last_id + 1

print(f"Processed all bags and saved in {output_folder}")